In [ ]:
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = next((c for c in [cwd, *cwd.parents] if (c / "utilities" / "functions.py").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")
sys.path.append(str(repo_root))

import utilities.functions as functions

importlib.reload(functions)

data_root = repo_root / "ms0_5"

# The pair this notebook describes. Everything below is written per pair -- the tables and
# the three figures all carry PAIR in their file name -- so changing this string and running
# the notebook again produces the same set for another pair without overwriting the last one.
PROTEIN = 'RT'
PAIR = 'M41L-T215Y'
MIN_POS, MAX_POS = 39, 226

ALL_SEQ = functions.read_seq(str(data_root / "RT" / "data" / "rt.reduce4.seq"))
with open(str(data_root / "RT" / "data" / "rt.consensus.reduce4.seq")) as f:
    CONSENSUS = f.read().strip()
REDUX = functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2]."""
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


J = build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), MIN_POS, MAX_POS)
print(f"{PROTEIN}: {len(ALL_SEQ):,} sequences, positions {MIN_POS}-{MAX_POS}")

In [ ]:
# ---------------------------------------------------------------------------
# The same energies as the other notebooks, for one pair (p1: wt1->mt1, p2: wt2->mt2) on a
# given sequence background. The sequence is first rewritten to wild type at both positions,
# then
#     M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
# with S the background coupling sum over every other position, and
#     dE (first mutation only)  = M[wt1, wt2] - M[mt1, wt2]
#     dE (second mutation only) = M[wt1, wt2] - M[wt1, mt2]
#     dE double                 = M[wt1, wt2] - M[mt1, mt2]
# Positive means the mutation lowers the energy, i.e. is favourable on that background.
# ---------------------------------------------------------------------------
_AA_CODE = np.full(256, 4, dtype=np.uint8)
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    A = Jm[p].copy()
    A[list(excluded)] = 0.0
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def pair_energies(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    """dE of each single mutation and of the double, per sequence."""
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)
    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    base = M[:, wt1, wt2]
    return base - M[:, mt1, wt2], base - M[:, wt1, mt2], base - M[:, mt1, mt2]


print("machinery ready")

In [ ]:
# =============================================================================
# The three energies for every sequence that carries K122E-D123N.
#
# A carrier is a sequence with both mutant residues present in the reduced alphabet. For each
# of them the pair is scored on that sequence's own background, giving three numbers: what the
# first mutation alone would do, what the second alone would do, and what the two together do.
# =============================================================================
lab1, lab2 = functions.split_pairs(PAIR)
wt1, pos1, mt1 = functions.split_pair(functions.unreduced_to_reduced(REDUX, lab1))
wt2, pos2, mt2 = functions.split_pair(functions.unreduced_to_reduced(REDUX, lab2))
p1, p2 = pos1 - MIN_POS, pos2 - MIN_POS
iwt1, imt1 = "ABCD".index(wt1), "ABCD".index(mt1)
iwt2, imt2 = "ABCD".index(wt2), "ABCD".index(mt2)

CODES, ONEHOT = encode_seqs(ALL_SEQ, MIN_POS, MAX_POS)
de1, de2, de12 = pair_energies(ONEHOT, J, p1, p2, iwt1, imt1, iwt2, imt2)
carriers = (CODES[:, p1] == imt1) & (CODES[:, p2] == imt2)

COLS = [f'dE {lab1}', f'dE {lab2}', 'dE double']


def frame(mask):
    return pd.DataFrame({COLS[0]: de1[mask], COLS[1]: de2[mask], COLS[2]: de12[mask]})


DE = frame(carriers)                    # the sequences that carry the pair
DE_NON = frame(~carriers)               # every other background in the alignment
GROUPS = {'carriers': DE, 'non-carriers': DE_NON}

print(f"{PAIR}  =  {wt1}{pos1}{mt1}-{wt2}{pos2}{mt2} in the reduced alphabet")
print(f"{carriers.sum():,} carriers of {len(ALL_SEQ):,} sequences "
      f"({100 * carriers.mean():.1f}%)")
print(f"{(~carriers).sum():,} non-carriers")

In [ ]:
# =============================================================================
# The table behind the box plot: the quartiles each box is drawn from, plus the share of
# carriers on the favourable side of zero, and the two comparisons that decide the epistasis
# category (does the double beat each single?).
# =============================================================================
def describe(df, label):
    """Quartiles, spread and the share on the favourable side of zero, per quantity."""
    t = df.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
    t['IQR'] = t['75%'] - t['25%']
    t['% > 0'] = (100 * (df > 0).mean()).round(1)
    t['count'] = t['count'].astype(int)
    t = t.round(2)

    n = len(df)
    b1 = int((df[COLS[2]] > df[COLS[0]]).sum())
    b2 = int((df[COLS[2]] > df[COLS[1]]).sum())
    print(f"{label}: of {n:,}, dE double beats {COLS[0]} in {b1:,} ({100*b1/n:.1f}%) "
          f"and {COLS[1]} in {b2:,} ({100*b2/n:.1f}%)")
    t.to_csv(f'de_distribution_{PAIR}_{label.replace(" ", "_")}.csv')
    return t


SUMMARY = {label: describe(df, label) for label, df in GROUPS.items()}
summary = SUMMARY['carriers']
summary

In [ ]:
import matplotlib.pyplot as plt

# =============================================================================
# Box plot of the three distributions over the carriers.
#
# Boxes are the quartiles, the line inside is the median, the diamond is the mean, whiskers
# reach 1.5 x IQR and everything past them is drawn as a faint point. The dashed line at zero
# is the wild type: a distribution sitting left of it is unfavourable on those backgrounds.
# =============================================================================
SURFACE, INK, INK2, MUTED, GRID, BASE = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e6e5df', '#c3c2b7'
COLOURS = ['#1f77b4', '#ff7f0e', '#2ca02c']       # same hues the other figures use

def box_figure(df, label, colours=COLOURS):
    """One horizontal box per quantity, drawn the same way for either group."""
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    bp = ax.boxplot([df[c] for c in COLS], vert=False, widths=0.58, patch_artist=True,
                    showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor=SURFACE, markeredgecolor=INK,
                                   markersize=4.5, markeredgewidth=0.8),
                    medianprops=dict(color=INK, linewidth=1.2),
                    whiskerprops=dict(color=BASE, linewidth=1.0),
                    capprops=dict(color=BASE, linewidth=1.0),
                    flierprops=dict(marker='o', markersize=2.2, markerfacecolor=MUTED,
                                    markeredgecolor='none', alpha=0.35))
    for patch, colour in zip(bp['boxes'], colours):
        patch.set_facecolor(colour)
        patch.set_alpha(0.55)
        patch.set_edgecolor(colour)
        patch.set_linewidth(1.0)

    ax.axvline(0, color=BASE, ls=(0, (4, 3)), lw=0.8, zorder=0)
    ax.set_yticks([1, 2, 3], COLS, fontsize=9, color=INK2)
    ax.set_xlabel('ΔE  (positive = favourable on that background)', fontsize=9, color=INK2)
    ax.set_title(f'{PAIR}: the {len(df):,} {label}, scored on their own backgrounds',
                 fontsize=9.5, color=INK, loc='left', pad=8)
    ax.tick_params(length=0, labelsize=8)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(BASE)
    ax.grid(axis='x', color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)

    stem = f'de_boxplot_{PAIR}_{label.replace(" ", "_")}'
    fig.savefig(stem + '.png', dpi=400, bbox_inches='tight', pad_inches=0.02,
                facecolor=SURFACE)
    fig.savefig(stem + '.pdf', bbox_inches='tight', pad_inches=0.02, facecolor=SURFACE)
    print('saved', stem + '.png / .pdf')
    return fig, ax


box_figure(DE, 'carriers')
plt.show()

In [ ]:
# =============================================================================
# The same three distributions over the backgrounds that do NOT carry the pair -- every other
# sequence in the alignment. Same axes and same construction as the carrier figure, so the
# two can be read against each other: what the model says about this pair in the sequences
# where it is absent.
# =============================================================================
box_figure(DE_NON, 'non-carriers')
SUMMARY['non-carriers']

In [ ]:
# =============================================================================
# Both groups on one axis: for each quantity, the carriers above and the non-carriers below.
# Colour marks the group rather than the quantity here, since the group is the comparison.
# The gap between the two boxes of a row is what the background does to that quantity -- how
# far the sequences carrying the pair sit from the ones that do not.
# =============================================================================
CARRIER_C, OTHER_C = '#1f77b4', '#9e9e9e'
fig, ax = plt.subplots(figsize=(6.6, 3.4))
for k, (label, df, colour) in enumerate([('carriers', DE, CARRIER_C),
                                         ('non-carriers', DE_NON, OTHER_C)]):
    pos = [i + (0.19 if k == 0 else -0.19) for i in range(1, len(COLS) + 1)]
    bp = ax.boxplot([df[c] for c in COLS], positions=pos, vert=False, widths=0.32,
                    patch_artist=True, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor=SURFACE, markeredgecolor=INK,
                                   markersize=3.8, markeredgewidth=0.7),
                    medianprops=dict(color=INK, linewidth=1.1),
                    whiskerprops=dict(color=BASE, linewidth=0.9),
                    capprops=dict(color=BASE, linewidth=0.9),
                    flierprops=dict(marker='o', markersize=1.8, markerfacecolor=MUTED,
                                    markeredgecolor='none', alpha=0.25))
    for patch in bp['boxes']:
        patch.set_facecolor(colour)
        patch.set_alpha(0.6)
        patch.set_edgecolor(colour)
        patch.set_linewidth(1.0)
    ax.plot([], [], color=colour, lw=6, alpha=0.6, label=f'{label} ({len(df):,})')

ax.axvline(0, color=BASE, ls=(0, (4, 3)), lw=0.8, zorder=0)
ax.set_yticks(range(1, len(COLS) + 1), COLS, fontsize=9, color=INK2)
ax.set_ylim(0.5, len(COLS) + 0.5)
ax.set_xlabel('ΔE  (positive = favourable on that background)', fontsize=9, color=INK2)
ax.set_title(f'{PAIR}: carriers against every other background', fontsize=9.5, color=INK,
             loc='left', pad=18)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, fontsize=8,
          labelcolor=INK2, handlelength=1.1, handletextpad=0.4, columnspacing=1.2)
ax.tick_params(length=0, labelsize=8)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color(BASE)
ax.grid(axis='x', color=GRID, linewidth=0.6)
ax.set_axisbelow(True)

fig.savefig(f'de_boxplot_{PAIR}_both.png', dpi=400, bbox_inches='tight', pad_inches=0.02,
            facecolor=SURFACE)
fig.savefig(f'de_boxplot_{PAIR}_both.pdf', bbox_inches='tight', pad_inches=0.02,
            facecolor=SURFACE)
print(f'saved de_boxplot_{PAIR}_both.png / .pdf')
plt.show()